In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahasaad12/verilog-example/counter.v


# Mistral (Large Language Model)

In [2]:
!pip install transformers==4.52.4

In [3]:
from huggingface_hub import login
login()

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
with open("/kaggle/input/datasets/mahasaad12/counter-verilog/counter.v", "r") as f:
    rtl = f.read()

print(rtl)

module counter(
    input clk,
    input rst,
    output reg [3:0] count
);

always @(posedge clk)
begin
    if(rst)
        count<=0;
    else
        count<=count+1;
end



# verilog parser

In [10]:
!apt-get update
!apt-get install -y iverilog
!iverilog -V
!pip install pyverilog

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,812 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease  
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]  
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Ge

In [13]:
from pyverilog.vparser.parser import parse
from pyverilog.vparser.ast import *
file_path = "/kaggle/input/datasets/mahasaad12/counter-verilog/counter.v"


ast, directives = parse([file_path])

Generating LALR tables


# Extract model name 

In [14]:
from pyverilog.vparser.ast import ModuleDef

for description in ast.description.definitions:
    if isinstance(description, ModuleDef):
        print("Module:", description.name)

Module: counter


# Extract Ports

In [41]:
for description in ast.description.definitions:

    print("Inputs")

    for item in description.items:

        if isinstance(item, Decl):

            for obj in item.list:

                if isinstance(obj, Input):
                    print(obj.name)

    print()

    print("Outputs")

    for item in description.items:

        if isinstance(item, Decl):

            for obj in item.list:

                if isinstance(obj, Output):
                    print(obj.name)

Inputs

Outputs


# Extract Registers

In [17]:
for description in ast.description.definitions:

    print("Registers")

    for item in description.items:

        if isinstance(item, Decl):

            for obj in item.list:

                if isinstance(obj, Reg):
                    print(obj.name)

Registers


# Extract Wires

In [18]:
for description in ast.description.definitions:

    print("Wires")

    for item in description.items:

        if isinstance(item, Decl):

            for obj in item.list:

                if isinstance(obj, Wire):
                    print(obj.name)

Wires


# Count Always Blocks

In [19]:
from pyverilog.vparser.ast import Always

count = 0

for description in ast.description.definitions:

    for item in description.items:

        if isinstance(item, Always):
            count += 1

print("Always Blocks:", count)

Always Blocks: 1


# Detect Sequential Logic

In [20]:
for description in ast.description.definitions:

    for item in description.items:

        if isinstance(item, Always):

            print(item.sens_list)

# Store Everything in a Dictionary

In [52]:
rtl_info = {

    "module": "",

    "inputs": [],

    "outputs": [],

    "registers": [],

    "wires": [],

    "always_blocks": 0

}
print(rtl_info)

{'module': '', 'inputs': [], 'outputs': [], 'registers': [], 'wires': [], 'always_blocks': 0}


In [54]:
# Custom AST visitor
class RTLVisitor:

    def visit(self, node):

        method_name = "visit_" + node.__class__.__name__

        visitor = getattr(
            self,
            method_name,
            self.generic_visit
        )

        visitor(node)


    def generic_visit(self, node):

        for child in node.children():
            self.visit(child)


    def visit_ModuleDef(self, node):

        rtl_info["module"] = node.name

        self.generic_visit(node)


    def visit_Decl(self, node):

        for item in node.list:

            if isinstance(item, Input):
                rtl_info["inputs"].append(item.name)

            elif isinstance(item, Output):
                rtl_info["outputs"].append(item.name)

            elif isinstance(item, Reg):
                rtl_info["registers"].append(item.name)

            elif isinstance(item, Wire):
                rtl_info["wires"].append(item.name)


        self.generic_visit(node)


    def visit_Always(self, node):

        rtl_info["always_blocks"] += 1

        self.generic_visit(node)



# Run parser
visitor = RTLVisitor()

visitor.visit(ast)


print(rtl_info)

{'module': 'counter', 'inputs': [], 'outputs': [], 'registers': [], 'wires': [], 'always_blocks': 1}


In [ ]:
visitor = RTLVisitor()

visitor.visit(ast)

In [ ]:
print(rtl_info)

In [49]:
rtl_info["module"] = "counter"

rtl_info["inputs"].append("clk")
rtl_info["inputs"].append("rst")

rtl_info["outputs"].append("count")

rtl_info["registers"].append("count")

rtl_info["always_blocks"] = 1

In [50]:
print(rtl_info)

{'module': 'counter', 'inputs': ['clk', 'rst'], 'outputs': ['count'], 'registers': ['count'], 'wires': [], 'always_blocks': 1}


# Save as JSON

In [57]:
with open("rtl_info.json", "w") as file:
    json.dump(rtl_info, file, indent=4)

# prompt